In [ ]:
import json
import sys
from dataclasses import replace
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "workers" / "extraction").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from workers.extraction import Settings

settings = replace(
    Settings.from_env(),
    llm_base_url="http://localhost:1234/v1",     
    llm_model="bielik-4.5b-v3.0-instruct",
)
print(settings)

NAME = "patient01_description"         
IMAGE = ROOT / "data" / "reports" / f"{NAME}.jpg"
GOLD_DIR = ROOT / "data" / "ground_truth"
print(IMAGE, IMAGE.exists())

In [ ]:
from PIL import Image, ImageOps

img = ImageOps.exif_transpose(Image.open(IMAGE))
img.thumbnail((700, 700))
img

In [ ]:
import tempfile

from workers.extraction.orientation import get_orientation_detector, make_upright

orientation = get_orientation_detector(settings.ocr_orientation)   # ładuje model, chwilę trwa

with tempfile.TemporaryDirectory() as tmp:
    upright_path, angle = make_upright(IMAGE, orientation, Path(tmp))   # to samo, co robi pipeline
    with Image.open(upright_path) as im:
        fixed = im.copy()

print("kąt korekty:", angle)
fixed.thumbnail((700, 700))
fixed

In [ ]:
from workers.extraction.ocr import get_ocr_engine
from workers.extraction.pipeline import run_ocr

ocr = get_ocr_engine(settings.ocr_engine, lang=settings.ocr_lang) 
ocr_result = run_ocr([IMAGE], ocr, orientation)

print(f"silnik: {ocr_result.engine} {ocr_result.engine_version}, czas: {ocr_result.seconds:.1f}s")
print(f"średnia pewność: {ocr_result.mean_confidence:.3f}, obrót stron: {[p.rotation for p in ocr_result.pages]}")
print()
raw_text = ocr_result.text
print(raw_text)

In [ ]:
lines = [line for page in ocr_result.pages for line in page.lines]
for line in sorted(lines, key=lambda l: l.confidence or 0)[:10]:
    print(f"{line.confidence:.3f}  {line.text}")

In [ ]:
import importlib
from difflib import SequenceMatcher

import workers.extraction.evaluation.compare as compare
from workers.extraction.normalize import normalize_text

importlib.reload(compare)         

text = normalize_text(raw_text)
print(f"przed: {len(raw_text)} znaków, po: {len(text)} znaków\n")
print(text)

reference_path = GOLD_DIR / "text" / f"{NAME}.txt"
if reference_path.exists():
    reference = reference_path.read_text(encoding="utf-8")
    cer_raw = compare.char_error_rate(reference, raw_text, normalize=False)
    cer_norm = compare.char_error_rate(reference, raw_text)
    print(f"\nCER bez normalizacji: {cer_raw:.1%}")
    print(f"CER z normalizacją:   {cer_norm:.1%}")
    print(f"normalizacja usunęła {cer_raw - cer_norm:.1%} punktu błędu\n")

  
    ref_words = normalize_text(reference).split()
    ocr_words = text.split()
    diffs = [
        op for op in SequenceMatcher(None, ref_words, ocr_words, autojunk=False).get_opcodes()
        if op[0] != "equal"
    ]
    labels = {"replace": "zamiana", "delete": "brak w OCR", "insert": "nadmiar w OCR"}
    print(f"różnice w treści: {len(diffs)}")
    for tag, i1, i2, j1, j2 in diffs:
        print(f"  {labels[tag]:<14} transkrypcja: {' '.join(ref_words[i1:i2])!r:<40} "
              f"ocr: {' '.join(ocr_words[j1:j2])!r}")
else:
    print(f"\nBrak ręcznej transkrypcji: {reference_path}")

In [ ]:
from workers.extraction.schema import llm_json_schema
from workers.extraction.structuring.prompting import PROMPT_VERSION, build_messages

messages = build_messages(text, few_shot=settings.few_shot_examples)
schema = llm_json_schema() if settings.llm_structured_output else None

print(f"wersja promptu: {PROMPT_VERSION}, wiadomości: {len(messages)}, "
      f"structured output: {schema is not None}\n")
for m in messages:
    print(f"===== {m['role']} ({len(m['content'])} znaków) =====")
    print(m["content"])
    print()

In [ ]:
import json

from workers.extraction.llm import LLMError, get_llm_client

llm = get_llm_client(settings)
print(f"dostawca: {llm.provider}, model: {llm.model}, serwer: {settings.llm_base_url}")

try:
    response = llm.complete(messages, json_schema=schema)
except LLMError as exc:
    print("Błąd LLM:", exc)
else:
    print(f"czas: {response.seconds:.1f}s")
    print(f"tokeny: {response.usage}\n")
    try:
        print(json.dumps(json.loads(response.text), ensure_ascii=False, indent=2))
    except json.JSONDecodeError as exc:
        print(f"(odpowiedź nie jest poprawnym JSON-em: {exc}, surowy tekst poniżej)\n")
        print(response.text)

In [ ]:
from workers.extraction.structuring.parsing import ParseError, parse_response

try:
    data = parse_response(response.text)
except ParseError as exc:
    data = None
    print("Błąd parsowania:", exc)
else:
    print(f"OK: {type(data).__name__}")
    print(f"modalności: {[m.value for m in data.modalnosci]}")
    print(f"zmian: {len(data.zmiany)}, węzłów: {len(data.wezly)}")
    print(f"BI-RADS wniosku: {data.wniosek.birads.value if data.wniosek.birads else None}")

In [ ]:
import pandas as pd

def val(x):
    """Enum -> jego wartość tekstowa, reszta bez zmian."""
    return getattr(x, "value", x)

obs_rows = []
for z in data.zmiany:
    for obs in z.obserwacje:
        obs_rows.append({
            "zmiana": z.id,
            "strona": val(z.strona),
            "modalność": val(obs.modalnosc),
            "lokalizacje": ", ".join(obs.lokalizacje),
            "godzina": obs.godzina,
            "od brodawki [mm]": obs.odleglosc_od_brodawki and obs.odleglosc_od_brodawki.mm(),
            "wymiary": obs.wymiary and f"{obs.wymiary.wartosci} {val(obs.wymiary.jednostka)} ({val(obs.wymiary.kwalifikator)})",
            "max [mm]": obs.wymiary and obs.wymiary.max_mm(),
            "głębokość [mm]": obs.glebokosc and obs.glebokosc.mm(),
            "kształt": val(obs.ksztalt),
            "brzeg": val(obs.brzeg),
            "echogeniczność": val(obs.echogenicznosc),
            "jednorodność": val(obs.jednorodnosc),
            "cytat": obs.cytat,
        })

print("ZMIANY (kolumna = jedna obserwacja)")
display(pd.DataFrame(obs_rows).T)

for z in data.zmiany:
    if z.polaczenie_obserwacji:
        p = z.polaczenie_obserwacji
        print(f"zmiana {z.id}: połączenie pewne={p.pewne}, powód: {p.powod}")

print("\nWĘZŁY")
display(pd.DataFrame([
    {
        "modalność": val(w.modalnosc),
        "strona": val(w.strona),
        "okolica": w.okolica,
        "patologiczne": w.patologiczne,
        "cytat": w.cytat,
    }
    for w in data.wezly
]))

print("\nPIERSI")
display(pd.DataFrame({
    side: {
        "budowa": p.budowa,
        "zwapnienia": p.zwapnienia,
        "mikrozwapnienia": p.mikrozwapnienia,
        "zmiany ogniskowe": p.zmiany_ogniskowe,
        "skóra": p.skora.stan,
    }
    for side in ("lewa", "prawa")
    for p in [getattr(data.piersi, side)]
}))

print(f"\nWNIOSEK: BI-RADS {val(data.wniosek.birads)} | {data.wniosek.tekst} | zalecenie: {data.wniosek.zalecenie}")

In [ ]:
from workers.extraction.structuring.checks import check_extraction

warnings = check_extraction(data, text)

print(f"ostrzeżeń: {len(warnings)}\n")
for w in warnings:
    print(" -", w)

In [ ]:
from workers.extraction.normalize import fold, quote_similarity

haystack = fold(text)
for word in ["szczoteczkowate", "tłuszczowe", "złośliwości", "malign", "ogniskow", "ogniskowa"]:
    print(f"{word:<16} {quote_similarity(word, haystack):.2f}")

mmg_quote = fold(data.zmiany[0].obserwacje[0].cytat)
print("\n'nieregular' w cytacie MMG:", "nieregular" in mmg_quote)
print("'nieostr' w cytacie MMG:   ", "nieostr" in mmg_quote)

In [ ]:
from datetime import UTC, datetime

from workers.extraction.evaluation import evaluate, format_report, load_gold, load_transcripts
from workers.extraction.schema import ExtractionResult, Provenance
from workers.extraction.structuring.prompting import PROMPT_VERSION

gold = load_gold(GOLD_DIR)               
transcripts = load_transcripts(GOLD_DIR)


result_manual = ExtractionResult(
    source=IMAGE.name,
    status="ok",
    data=data,
    warnings=warnings,
    text=text,
    provenance=Provenance(
        ocr_engine=ocr_result.engine,
        ocr_engine_version=ocr_result.engine_version,
        llm_provider=llm.provider,
        llm_model=llm.model,
        prompt_version=PROMPT_VERSION,
    ),
    created_at=datetime.now(UTC),
)

report = evaluate(gold, {NAME: result_manual}, transcripts=transcripts)
print(format_report(report))

In [ ]:
import pandas as pd

from workers.extraction.evaluation.scores import SUMMARY_GROUPS

if "report" not in globals():
    raise RuntimeError("Najpierw uruchom komórkę z evaluate(...), bo `report` jeszcze nie istnieje.")

PCT_COLS = ["accuracy", "precision", "recall", "f1"]
COUNT_COLS = ["n", "wrong", "missed", "spurious", "gold", "pred", "match"]
SCORE_COLS = ["n", *PCT_COLS, "wrong", "missed", "spurious"]


def score_row(s):
    return {"n": s.n, "accuracy": s.accuracy, "precision": s.precision, "recall": s.recall,
            "f1": s.f1, "wrong": s.wrong, "missed": s.missed, "spurious": s.spurious}


def show(df, caption):
    df = df.copy()
    for c in df.columns:
        if c in COUNT_COLS:
            df[c] = df[c].astype(int)
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        fmt = {c: ("{:.2%}" if c == "CER" else "{:.2f}") for c in df.columns if c not in COUNT_COLS}
    display(df.style.format(fmt, na_rep="–").set_caption(caption)
              .set_table_styles([{"selector": "caption",
                                  "props": "font-weight: bold; font-size: 1.1em; text-align: left;"}]))



info = {
    "model": ", ".join(p["llm_model"] or "?" for p in report.provenance),
    "prompt": ", ".join(p["prompt_version"] for p in report.provenance),
    "schema": ", ".join(str(p["schema_version"]) for p in report.provenance),
    "ocenione": len(report.evaluated),
    "błędy (liczone jako puste)": len(report.errors),
    **{f"pominięte: {reason}": len(keys) for reason, keys in report.skipped.items()},
}
display(pd.Series(info, name="wartość").to_frame().style.set_caption("PRZEBIEG"))


if report.ocr_cer:
    show(pd.Series(report.ocr_cer, name="CER").to_frame(), "OCR – CER na dokument")


groups_df = pd.DataFrame(
    {g or "wszystkie pola": score_row(report.group_score(g)) for g in (None, *SUMMARY_GROUPS)}
).T[SCORE_COLS]
show(groups_df, "GRUPY PÓL")


entities_df = pd.DataFrame(
    {name: {"gold": e.gold, "pred": e.predicted, "match": e.matched,
            "precision": e.precision, "recall": e.recall, "f1": e.f1}
     for name, e in report.entities.items()}
).T
show(entities_df, "ENCJE")


fields_df = pd.DataFrame({name: score_row(s) for name, s in sorted(report.fields.items())}).T[SCORE_COLS]
show(fields_df, "WSZYSTKIE POLA")


errors_df = fields_df[fields_df["accuracy"] < 1].sort_values(["accuracy", "n"], ascending=[True, False])
show(errors_df, f"POLA Z BŁĘDAMI ({len(errors_df)})")